## Phase 3 — Week 1: LLM Fundamentals

### Learning objectives:
- Understand how LLMs work (tokens, context, temperature)
- Make first Groq API call
- Understand system vs user prompts
- Build a fraud alert summarizer using prompt engineering

In [1]:
# ============================================================
# CELL 1: First LLM API Call
# ============================================================

import os
from groq import Groq

# Initialize client — reads API key from environment variable
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

# Make your first API call
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",  # fast, free LLaMA model
    messages=[
        {
            "role": "system",
            "content": "You are a helpful fraud analyst assistant at a bank."
        },
        {
            "role": "user", 
            "content": "What are the top 3 signs that a credit card transaction might be fraudulent?"
        }
    ],
    temperature=0.3,  # low temperature for consistent factual output
    max_tokens=300    # limit response length
)

# Extract and print the response
print(response.choices[0].message.content)

# Print token usage
print(f"\n--- Token Usage ---")
print(f"Prompt tokens:     {response.usage.prompt_tokens}")
print(f"Completion tokens: {response.usage.completion_tokens}")
print(f"Total tokens:      {response.usage.total_tokens}")

As a fraud analyst assistant, I've identified the top 3 signs that a credit card transaction might be fraudulent:

1. **Geographic Discrepancy**: If a credit card transaction is made from a location that is significantly far away from the cardholder's usual geographic area, it may indicate a fraudulent transaction. This could be due to the card being used by an unauthorized person or a stolen card being used in a different location. We use geolocation data and cardholder behavior patterns to identify suspicious transactions.

2. **Unusual Transaction Amounts or Patterns**: If a credit card transaction is significantly higher or lower than the cardholder's usual spending habits, it may indicate a fraudulent transaction. We also look for unusual patterns, such as multiple transactions in a short period or transactions that are not consistent with the cardholder's typical spending habits.

3. **Device or Browser Anomalies**: If a credit card transaction is made from a device or browser th

In [2]:
# ============================================================
# CELL 2: Temperature Experiment
# ============================================================

def ask_llm(question, temperature):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful fraud analyst assistant at a bank."
            },
            {
                "role": "user",
                "content": question
            }
        ],
        temperature=temperature,
        max_tokens=150
    )
    return response.choices[0].message.content

question = "In one sentence, what is the biggest sign of credit card fraud?"

print("=== Temperature 0.0 (deterministic) ===")
print(ask_llm(question, 0.0))

print("\n=== Temperature 0.0 (run again - should be identical) ===")
print(ask_llm(question, 0.0))

print("\n=== Temperature 1.0 (creative) ===")
print(ask_llm(question, 1.0))

print("\n=== Temperature 1.0 (run again - will differ) ===")
print(ask_llm(question, 1.0))

=== Temperature 0.0 (deterministic) ===
The biggest sign of credit card fraud is a sudden and unexplained increase in transactions, especially if they are international, online, or involve high-value purchases, which can indicate that the card has been compromised or stolen.

=== Temperature 0.0 (run again - should be identical) ===
The biggest sign of credit card fraud is a sudden and unexplained increase in transactions, especially if they are international, online, or involve high-value purchases, which can indicate that the card has been compromised or stolen.

=== Temperature 1.0 (creative) ===
The biggest sign of credit card fraud is a sudden and unexplained change in spending behavior, such as frequent or high-value transactions originating from unfamiliar locations or IP addresses, often coupled with an increase in declined or authorized transactions that the cardholder didn't make.

=== Temperature 1.0 (run again - will differ) ===
The biggest sign of credit card fraud is a su

In [3]:
# ============================================================
# CELL 3: Fraud Alert Summarizer
# ============================================================

def generate_fraud_summary(transaction):
    """
    Takes a transaction dictionary and generates
    a professional fraud investigation summary.
    """
    
    # Format transaction data as structured text
    transaction_text = f"""
    Transaction ID: {transaction['id']}
    Amount: ${transaction['amount']:,.2f}
    Merchant: {transaction['merchant']}
    Location: {transaction['location']}
    Time: {transaction['time']}
    Card Type: {transaction['card_type']}
    Previous avg spend: ${transaction['avg_spend']:,.2f}
    Fraud Score: {transaction['fraud_score']:.2%}
    """
    
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "system",
                "content": """You are a senior fraud analyst at a bank. 
                When given transaction details, you write concise 
                professional investigation summaries.
                
                Your summaries must include:
                1. Risk assessment (High/Medium/Low)
                2. Key red flags identified
                3. Recommended action
                
                Keep summaries under 150 words. Be direct and factual."""
            },
            {
                "role": "user",
                "content": f"Generate a fraud investigation summary for this transaction:\n{transaction_text}"
            }
        ],
        temperature=0.1,
        max_tokens=200
    )
    
    return response.choices[0].message.content

# Test with a suspicious transaction
suspicious_transaction = {
    "id": "TXN-2024-88821",
    "amount": 4500.00,
    "merchant": "Electronics Plus",
    "location": "Lagos, Nigeria",
    "time": "02:47 AM",
    "card_type": "Visa Debit",
    "avg_spend": 85.00,
    "fraud_score": 0.94
}

# Test with a normal transaction
normal_transaction = {
    "id": "TXN-2024-88822",
    "amount": 42.50,
    "merchant": "Starbucks",
    "location": "New York, US",
    "time": "08:15 AM",
    "card_type": "Mastercard Credit",
    "avg_spend": 38.00,
    "fraud_score": 0.03
}

print("=== SUSPICIOUS TRANSACTION ===")
print(generate_fraud_summary(suspicious_transaction))

print("\n=== NORMAL TRANSACTION ===")
print(generate_fraud_summary(normal_transaction))

=== SUSPICIOUS TRANSACTION ===
**Fraud Investigation Summary**

**Transaction ID:** TXN-2024-88821
**Risk Assessment:** High

**Key Red Flags Identified:**

- Excessive transaction amount ($4,500.00) significantly exceeding the customer's previous average spend ($85.00).
- High Fraud Score (94.00%).
- Unusual transaction time (02:47 AM).
- Merchant located in Lagos, Nigeria, which is a high-risk country for card-not-present (CNP) transactions.

**Recommended Action:** Flag the transaction for further review and potential reversal. Contact the cardholder to verify the transaction and gather additional information.

=== NORMAL TRANSACTION ===
**Fraud Investigation Summary**

**Transaction ID:** TXN-2024-88822
**Risk Assessment:** Low
**Key Red Flags Identified:** None
**Recommended Action:** No further action required.

The transaction amount of $42.50 is within the customer's average spend of $38.00, indicating a normal transaction pattern. The merchant is a reputable Starbucks location

## Day 2 — Prompt Engineering

Four core techniques:
1. Zero-shot — no examples
2. Few-shot — examples guide format and reasoning
3. Chain of thought — model explains its reasoning
4. Structured JSON output — machine-parseable responses

In [4]:
# ============================================================
# CELL 4: Zero-shot vs Few-shot Prompting
# ============================================================

# --- Zero-shot: no examples given ---

zero_shot_response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "system",
            "content": "You are a fraud risk classifier. Classify transactions as HIGH, MEDIUM, or LOW risk."
        },
        {
            "role": "user",
            "content": "Transaction: $3,200 at a jewellery store at 3am from a customer whose average spend is $50."
        }
    ],
    temperature=0.0,
    max_tokens=100
)
print("=== Zero-shot Response ===")
print(zero_shot_response.choices[0].message.content)

=== Zero-shot Response ===
Based on the provided information, I would classify this transaction as HIGH risk.

Reasons for classification:

1. **Unusual time of purchase**: 3am is an unusual time for a transaction, especially for a high-value purchase like $3,200. This could indicate a potential fraudulent activity.
2. **Significant deviation from average spend**: The customer's average spend is $50, but the current transaction is $3,200, which is a significant deviation from their usual spending behavior.


In [5]:
# --- Few-shot: examples guide the model ---
few_shot_response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "system",
            "content": """You are a fraud risk classifier.
Classify transactions as HIGH, MEDIUM, or LOW risk.
Always respond in exactly this format:
RISK: [HIGH/MEDIUM/LOW]
REASON: [one sentence]

Examples:

Transaction: $45 at Starbucks at 8am, avg spend $40
RISK: LOW
REASON: Amount consistent with average spend at typical merchant and time.

Transaction: $8,000 electronics purchase at 2am, avg spend $100
RISK: HIGH
REASON: Amount 80x above average spend at unusual hour suggests card compromise.

Transaction: $200 restaurant dinner at 7pm, avg spend $60
RISK: MEDIUM
REASON: Amount above average but merchant and time are plausible."""
        },
        {
           "role": "user",
            "content": "Transaction: $3,200 at a jewellery store at 3am from a customer whose average spend is $50." 
        }
    ],
    temperature=0.0,
    max_tokens=100
)
print("\n=== Few-shot Response ===")
print(few_shot_response.choices[0].message.content)


=== Few-shot Response ===
RISK: HIGH
REASON: Amount 64x above average spend at unusual hour at a high-value merchant suggests card compromise.


In [6]:
# ============================================================
# CELL 5: Chain of Thought Prompting
# ============================================================

# Standard prompt - just asks for answer
standard_response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "system",
            "content": "You are a fraud analyst. Classify transactions as HIGH, MEDIUM, or LOW risk."
        },
        {
          "role": "user",
            "content": """Transaction details:
- Amount: $750
- Merchant: Amazon
- Time: 11pm
- Location: Customer's home city
- Average spend: $200
- Last transaction: 2 hours ago for $45
- Card used: Same device as usual

Risk classification?"""  
        }
    ],
    temperature = 0.0,
    max_tokens = 50
)
print("=== Standard Prompt ===")
print(standard_response.choices[0].message.content)

=== Standard Prompt ===
Based on the provided transaction details, I would classify this transaction as LOW risk. 

Reasons for this classification:

- The amount is within the customer's average spend range ($200).
- The merchant is Amazon, a reputable and well-known online retailer


In [7]:
# Chain of thought - ask model to reason step by step

cot_response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "system",
            "content": """You are a fraud analyst. 
When classifying transactions, think through each factor 
step by step before giving your final classification.

Format:
ANALYSIS:
- Amount check: [observation]
- Time check: [observation]  
- Location check: [observation]
- Device check: [observation]
- Pattern check: [observation]

FINAL RISK: [HIGH/MEDIUM/LOW]
CONFIDENCE: [HIGH/MEDIUM/LOW]"""
        },
        {
            "role": "user",
            "content": """Transaction details:
- Amount: $750
- Merchant: Amazon
- Time: 11pm
- Location: Customer's home city
- Average spend: $200
- Last transaction: 2 hours ago for $45
- Card used: Same device as usual

Analyze this transaction step by step."""
        }
    ],
    temperature=0.0,
    max_tokens=250
)

print("\n=== Chain of Thought Prompt ===")
print(cot_response.choices[0].message.content)


=== Chain of Thought Prompt ===
ANALYSIS:
- Amount check: The transaction amount of $750 is significantly higher than the customer's average spend of $200. This could be a potential red flag.
- Time check: The transaction occurred at 11pm, which is outside of the typical business hours for Amazon. This could indicate a potential unauthorized transaction or a purchase made under unusual circumstances.
- Location check: The transaction occurred in the customer's home city, which is consistent with their usual location. This reduces the likelihood of a transaction being made in a foreign location.
- Device check: The card used was the same device as usual, which suggests that the customer is using their usual payment method. This reduces the likelihood of a transaction being made with a stolen or compromised card.
- Pattern check: The customer's last transaction was 2 hours ago for $45, which is a relatively small amount. This suggests that the customer may have been making legitimate pu

In [8]:
# ============================================================
# CELL 6: Structured JSON Output
# ============================================================

import json

def classify_transaction_json(transaction_details):
    """
    Returns a structured JSON fraud classification
    that can be directly used by downstream systems.    
    """

    response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "system",
                "content": """You are a fraud classification system.
Analyze transactions and respond ONLY with a JSON object.
No explanation, no markdown, no extra text — pure JSON only.

Required JSON format:
{
    "transaction_id": "string",
    "risk_level": "HIGH" or "MEDIUM" or "LOW",
    "confidence": "HIGH" or "MEDIUM" or "LOW",
    "fraud_probability": float between 0.0 and 1.0,
    "red_flags": ["list", "of", "flags"],
    "recommended_action": "string",
    "requires_human_review": true or false
}"""
                },
            {
                "role": "user",
                "content": f"Classify this transaction:\n{transaction_details}"
            }
        ],
        temperature=0.0,
        max_tokens=300
    )

    raw_response = response.choices[0].message.content

    # Parse JSON safely
    try:
        parsed = json.loads(raw_response)
        return parsed
    except json.JSONDecodeError:
        return {"error": "Failed to parse JSON", "raw": raw_response}

# Test transaction
transaction = """
Transaction ID: TXN-2024-99901
Amount: $5,800
Merchant: Western Union
Time: 1:23 AM
Location: Different city from customer home
Average spend: $120
Last 3 transactions: $45, $32, $28 (all today)
Device: New device never seen before
"""

result = classify_transaction_json(transaction)

print("=== Raw JSON Output ===")
print(json.dumps(result, indent=2))

print("\n=== Accessing Individual Fields ===")
print(f"Risk Level:       {result.get('risk_level')}")
print(f"Fraud Probability: {result.get('fraud_probability')}")
print(f"Requires Review:  {result.get('requires_human_review')}")
print(f"Red Flags:")
for flag in result.get('red_flags', []):
    print(f"  - {flag}")
print(f"Action:           {result.get('recommended_action')}")

=== Raw JSON Output ===
{
  "transaction_id": "TXN-2024-99901",
  "risk_level": "HIGH",
  "confidence": "HIGH",
  "fraud_probability": 0.87,
  "red_flags": [
    "high_amount",
    "unusual_location",
    "new_device",
    "unusual_spend_pattern"
  ],
  "recommended_action": "Verify customer identity and location",
  "requires_human_review": true
}

=== Accessing Individual Fields ===
Risk Level:       HIGH
Fraud Probability: 0.87
Requires Review:  True
Red Flags:
  - high_amount
  - unusual_location
  - new_device
  - unusual_spend_pattern
Action:           Verify customer identity and location


# ============================================================
# CELL 7: Complete Fraud Analysis Pipeline
# ============================================================

In [12]:
def analyse_transaction(transaction_data):
    """
    Complete fraud analysis pipeline combining:
    1. JSON structured output
    2. Few-shot formating
    3. Chain of thought reasoning
    4. Safe JSON parsing
    """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "system",
                "content": """You are an automated fraud detection system 
for a bank. Analyze each transaction thoroughly and respond 
ONLY with a JSON object — no markdown, no extra text.

Think through each factor before concluding:
- Amount vs average spend deviation
- Time of day risk
- Location consistency  
- Device familiarity
- Recent transaction patterns
- Merchant category risk

Required JSON format:
{
    "transaction_id": "string",
    "risk_level": "HIGH" or "MEDIUM" or "LOW",
    "confidence": "HIGH" or "MEDIUM" or "LOW",
    "fraud_probability": float 0.0-1.0,
    "red_flags": ["list of flags found"],
    "positive_signals": ["list of low-risk signals"],
    "recommended_action": "string",
    "requires_human_review": boolean,
    "priority": integer 1-5 where 5 is most urgent
}

Example output for a HIGH risk transaction:
{
    "transaction_id": "TXN-001",
    "risk_level": "HIGH",
    "confidence": "HIGH", 
    "fraud_probability": 0.92,
    "red_flags": ["amount_50x_above_average", "new_device", "3am_transaction"],
    "positive_signals": [],
    "recommended_action": "Block transaction and contact customer immediately",
    "requires_human_review": true,
    "priority": 5
}"""
            },
            {
                "role": "user",
                "content": f"Analyze this transaction:\n{transaction_data}"
            }
        ],
        temperature = 0.0,
        max_tokens=400
    )

    raw = response.choices[0].message.content

    try:
        result = json.loads(raw)
        return result
    except json.JSONDecodeError:
        # Clean common LLM formatting issues
        cleaned = raw.strip()
        if cleaned.startsith("''''"):
            cleaned = cleaned.split("''''")[1]
            if cleaned.startsith("json"):
                cleaned = cleaned[4:]
        try:
            return json.loads(cleaned.strip())
        except:
            return {"error": "Parse failed", "raw": raw}


# Run three transactions
transactions = [
    {
        "id": "TXN-A001",
        "details": "Amount: $9,500 | Merchant: Wire Transfer | Time: 4:15 AM | Location: Russia | Avg spend: $95 | Device: New | Recent: $22, $18, $31"
    },
    {
        "id": "TXN-A002", 
        "details": "Amount: $65 | Merchant: Uber | Time: 6:30 PM | Location: Home city | Avg spend: $55 | Device: Known | Recent: $45, $72, $38"
    },
    {
        "id": "TXN-A003",
        "details": "Amount: $380 | Merchant: Apple Store | Time: 2:00 PM | Location: Home city | Avg spend: $150 | Device: Known | Recent: $120, $89, $200"
    }
]


print("=" * 60)
print("FRAUD ANALYSIS PIPELINE — BATCH RESULTS")
print("=" * 60)


for txn in transactions:
    input_text = f"Transaction ID: {txn['id']}\n{txn['details']}"
    result = analyse_transaction(input_text)

    print(f"\nTransaction: {txn['id']}")
    print(f"Risk:        {result.get('risk_level')} "
          f"(confidence: {result.get('confidence')})")
    print(f"Probability: {result.get('fraud_probability')}")
    print(f"Priority:    {result.get('priority')}/5")
    print(f"Red flags:   {result.get('red_flags')}")
    print(f"Action:      {result.get('recommended_action')}")
    print("-" * 40)

FRAUD ANALYSIS PIPELINE — BATCH RESULTS

Transaction: TXN-A001
Risk:        HIGH (confidence: HIGH)
Probability: 0.85
Priority:    5/5
Red flags:   ['amount_50x_above_average', 'new_device', '4am_transaction', 'location_russia']
Action:      Block transaction and contact customer immediately
----------------------------------------

Transaction: TXN-A002
Risk:        LOW (confidence: HIGH)
Probability: 0.02
Priority:    1/5
Red flags:   []
Action:      Approve transaction
----------------------------------------

Transaction: TXN-A003
Risk:        LOW (confidence: HIGH)
Probability: 0.02
Priority:    1/5
Red flags:   []
Action:      Approve transaction
----------------------------------------


### Day 2 Summary — Prompt Engineering Techniques

| Technique | Use case | Key benefit |
|---|---|---|
| Zero-shot | Simple classification | Fast, minimal tokens |
| Few-shot | Consistent formatting | Reliable output structure |
| Chain of thought | Complex multi-factor analysis | Accuracy + explainability |
| JSON output | Production systems | Machine-parseable, integrable |

**Project 2 complete: Fraud Analysis Pipeline**
- Processes batch transactions automatically
- Returns structured JSON for downstream systems
- Priority scoring for analyst triage
- Robust error handling for production reliability

Files: phase3_llm_fundamentals.ipynb